# 08 Benchmark Comparison v1

This notebook orchestrates the unified monthly benchmark comparison.

Design choice:
- Keep `run_benchmark_comparison_v1.py` as the production source of truth.
- Use this notebook as a research-facing driver with clear cells for each benchmark.
- Add future benchmarks, such as `rl_overlay_ppo` or `rl_overlay_sac_changepoint`, by adding one loader cell and one entry in the `strategies` dictionary.

Current benchmark set:
- `equal_weight`
- `spy_buy_hold`
- `static_allocator`
- `rl_overlay_sac`

## 1. Imports and shared functions

The notebook imports the production script instead of duplicating benchmark logic. This keeps the notebook convenient for exploration while preserving one implementation path for reproducible runs.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from run_benchmark_comparison_v1 import (
    assemble_common_returns,
    build_equal_weight_strategy,
    compute_summary,
    load_spy_returns,
    load_strategy_backtest,
    parse_month_end,
    resolve_path,
    save_monthly_returns,
    save_plots,
    setup_logging,
    write_summary_text,
)

setup_logging()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

## 2. Configuration

Edit this cell when changing the test window, transaction costs, input files, or output directory. The default window below matches the current benchmark comparison run.

In [ ]:
PROJECT_ROOT = Path(".").resolve()

RETURNS_FILE = resolve_path(
    PROJECT_ROOT, "data/panel/monthly_stock_panel_basic_full.parquet"
)
SPY_FILE = resolve_path(PROJECT_ROOT, "data/benchmark/spy_monthly.parquet")
STATIC_BACKTEST_FILE = resolve_path(
    PROJECT_ROOT, "data/allocator/static_allocator_backtest.csv"
)
RL_BACKTEST_FILE = resolve_path(PROJECT_ROOT, "data/rl_overlay_sac/test_backtest.csv")
OUTDIR = resolve_path(PROJECT_ROOT, "data/benchmark_eval")
OUTDIR.mkdir(parents=True, exist_ok=True)

TEST_START = "2020-01"
TEST_END = "2024-11"
COST_BPS = 10.0
SPY_ENTRY_COST_BPS = 0.0

test_start = parse_month_end(TEST_START)
test_end = parse_month_end(TEST_END)

print(f"Project root: {PROJECT_ROOT}")
print(f"Test window: {TEST_START} to {TEST_END}")
print(f"Output directory: {OUTDIR}")

## 3. Equal-weight benchmark

The equal-weight strategy is built from the stock-level monthly panel. It uses the month-`t` investable universe, the panel's `target_ret_1m` as the month-`t` to `t+1` realized return, monthly rebalancing, and the same proportional transaction cost model.

In [ ]:
equal_weight = build_equal_weight_strategy(
    returns_file=RETURNS_FILE,
    test_start=test_start,
    test_end=test_end,
    cost_bps=COST_BPS,
)

display(equal_weight.head())
display(equal_weight.tail())

## 4. SPY buy-and-hold benchmark

SPY is loaded from `data/benchmark/spy_monthly.parquet` and uses the CRSP/WRDS `ret` column. There is no monthly rebalance. The optional one-time entry cost is applied later during the common-month assembly step.

In [1]:
spy_buy_hold = load_spy_returns(SPY_FILE)

display(spy_buy_hold.head())
display(spy_buy_hold.tail())

NameError: name 'load_spy_returns' is not defined

## 5. Static allocator benchmark

The static allocator is loaded from its existing backtest output. The loader uses `net_return` when available, so this benchmark keeps the transaction-cost treatment already computed by that strategy.

In [ ]:
static_allocator = load_strategy_backtest(
    STATIC_BACKTEST_FILE,
    strategy_name="static_allocator",
)

display(static_allocator.head())
display(static_allocator.tail())

## 6. RL overlay SAC benchmark

The SAC overlay is loaded from its existing test backtest output. As with the static allocator, the loader uses `net_return` when available.

In [ ]:
rl_overlay_sac = load_strategy_backtest(
    RL_BACKTEST_FILE,
    strategy_name="rl_overlay_sac",
)

display(rl_overlay_sac.head())
display(rl_overlay_sac.tail())

## 7. Assemble common monthly return table

This is the key integration point. Every strategy is restricted to the requested test window and then intersected to one common set of months. To add a new benchmark, create its DataFrame with `month_end`, `return`, and optional `turnover`, then add it to this dictionary.

In [ ]:
strategies = {
    "equal_weight": equal_weight,
    "spy_buy_hold": spy_buy_hold,
    "static_allocator": static_allocator,
    "rl_overlay_sac": rl_overlay_sac,
}

returns, turnover = assemble_common_returns(
    strategies=strategies,
    test_start=test_start,
    test_end=test_end,
    spy_entry_cost_bps=SPY_ENTRY_COST_BPS,
)

display(returns.head())
display(turnover.head())
print(f"Common months: {len(returns)}")
print(
    f"Common window: {returns['month_end'].min().date()} to {returns['month_end'].max().date()}"
)

## 8. Performance summary

The summary uses the same metrics as the production script: annualized return, annualized volatility, Sharpe ratio, max drawdown, cumulative return, mean monthly turnover, win rate, and number of months.

In [ ]:
summary = compute_summary(returns, turnover)

summary_display = summary.copy()
for col in [
    "annualized_return",
    "annualized_volatility",
    "max_drawdown",
    "cumulative_return",
    "mean_monthly_turnover",
    "win_rate",
]:
    summary_display[col] = summary_display[col].map(
        lambda x: "" if pd.isna(x) else f"{x:.2%}"
    )
summary_display["sharpe_ratio"] = summary_display["sharpe_ratio"].map(
    lambda x: "" if pd.isna(x) else f"{x:.3f}"
)

display(summary_display)

## 9. Save benchmark outputs

This cell writes the same CSV and TXT artifacts as the command-line script.

In [ ]:
monthly_returns_path = OUTDIR / "benchmark_monthly_returns.csv"
summary_csv_path = OUTDIR / "benchmark_summary.csv"
summary_txt_path = OUTDIR / "benchmark_summary.txt"

save_monthly_returns(returns, turnover, monthly_returns_path)
summary.to_csv(summary_csv_path, index=False)
write_summary_text(summary, summary_txt_path)

print(f"Saved: {monthly_returns_path}")
print(f"Saved: {summary_csv_path}")
print(f"Saved: {summary_txt_path}")

## 10. Save plots

The plotting function saves:
- `cumulative_nav_comparison.png`
- `monthly_return_comparison.png`
- `turnover_comparison.png`

In [ ]:
save_plots(returns, turnover, OUTDIR)

print(f"Saved plots under: {OUTDIR}")

## 11. Quick visual inspection inside notebook

These plots are for notebook review only. The production plot files are saved in the previous cell.

In [ ]:
nav = (1.0 + returns.set_index("month_end")).cumprod()
ax = nav.plot(figsize=(11, 5), title="Cumulative NAV Comparison")
ax.set_xlabel("Month")
ax.set_ylabel("NAV")
ax.grid(True, alpha=0.3)
plt.show()

ax = returns.set_index("month_end").plot(
    figsize=(11, 5), title="Monthly Return Comparison"
)
ax.axhline(0.0, color="black", linewidth=0.8, alpha=0.6)
ax.set_xlabel("Month")
ax.set_ylabel("Monthly return")
ax.grid(True, alpha=0.3)
plt.show()

turnover_cols = [
    col
    for col in ["equal_weight", "static_allocator", "rl_overlay_sac"]
    if turnover[col].notna().any()
]
ax = turnover.set_index("month_end")[turnover_cols].plot(
    figsize=(11, 5), title="Monthly Turnover Comparison"
)
ax.set_xlabel("Month")
ax.set_ylabel("Turnover")
ax.grid(True, alpha=0.3)
plt.show()

## 12. Extension pattern for future benchmarks

For `rl_overlay_ppo` or `rl_overlay_sac_changepoint`, the cleanest path is:

1. Save that strategy's backtest with at least `month_end` and `net_return`; include `turnover` if applicable.
2. Add one notebook cell:

```python
rl_overlay_ppo = load_strategy_backtest(
    PROJECT_ROOT / "data/rl_overlay_ppo/test_backtest.csv",
    strategy_name="rl_overlay_ppo",
)
```

3. Add it to `strategies`:

```python
strategies["rl_overlay_ppo"] = rl_overlay_ppo
```

If the new benchmark has a different schema, add a small loader function to `run_benchmark_comparison_v1.py` rather than rewriting logic in the notebook.